# Step-by-Step NER Finetuning Verification
This notebook walks through the internal components of the training script. Instead of running the full CLI black-box, we will load the dataset reader, initialize the pipeline, and fetch a batch to verify everything works end-to-end without crashing.

In [ ]:
import os
import sys
import torch
import confit
import edsnlp
from edsnlp.core.registries import registry
import itertools

# Move to the script's directory so relative paths in configs work
if os.path.basename(os.getcwd()) != "6_ner_finetuning":
    os.chdir('../scripts/6_ner_finetuning')
sys.path.append(os.getcwd())

print(f"Current working directory: {os.getcwd()}")

# Import your training script to register its plugins (like the new BigBio HF reader)
import train_ner_edsnlp

## 1. Test Dataset Configuration & Hugging Face Reader
Let's verify that the newly added BigBio HF reader successfully fetches the dataset and correctly parses the entities into spaCy documents.

In [ ]:
dataset_name = "spaccc"

# Fetch configuration for spaccc
cfg = train_ner_edsnlp.dataset_config(dataset_name)
print("Dataset config:", cfg)

# Initialize the registered BigBio reader
reader_factory = train_ner_edsnlp.read_bigbio_hf(
    dataset=cfg["hf_dataset"],
    config_name=cfg["hf_config"],
    split="train",
    span_setter=cfg["span_getter"]
)

# Fetch exactly one parsed document to inspect it
nlp_blank = edsnlp.blank("eds")
docs_generator = reader_factory(nlp_blank)
doc = next(docs_generator)

print("\n--- Document Text (Truncated) ---")
print(doc.text[:400] + "..." if len(doc.text) > 400 else doc.text)
print("\n--- Entities Found ---")
for ent in doc.ents:
    print(f"Span: '{ent.text}' | Label: {ent.label_} | Chars: [{ent.start_char}, {ent.end_char}]")

## 2. Initialize EDS-NLP Pipeline from Config
Parse `train_ner.cfg`, override the parameters, and resolve the pipeline mathematically. This step catches any missing configurations or paths.

In [ ]:
base_model = "../../models/deberta-v3-large"
config_path = "train_ner.cfg"

# Parse and override variables
config = confit.Config.from_disk(config_path)
config["vars"]["dataset"] = dataset_name
config["vars"]["base_model"] = base_model

print("Resolving pipeline...")
resolved = config.resolve(registry=registry)

# Extract the instantiated nlp pipeline
nlp = resolved["nlp"]
print("\nPipeline Components Active:")
for name, component in nlp.pipeline:
    print(f" - {name}: {type(component).__name__}")

## 3. Test DataLoader and Tensor Batching
We will substitute the dataloader's reader to fetch only 10 examples to prevent long loading times, then initialize the dataset and process a single PyTorch batch to inspect the generated tensors.

In [ ]:
print("Extracting Dataloader factory from resolved config...")
dataloader_factory = resolved["ner_train_dataloader"]

# Wrap the original reader to forcefully subset exactly 10 docs for fast inspection
original_reader = dataloader_factory.data[0]
def test_reader(pipeline):
    yield from itertools.islice(original_reader(pipeline), 10)
    
dataloader_factory.data = [test_reader]

print("Running initial document preprocessing and vocab building...")
train_dataloader = dataloader_factory(nlp)

print(f"\nDataloader created. Number of subsets/batches: {len(train_dataloader)}")

# Fetch 1 batch and inspect its tensors
batch = next(iter(train_dataloader))

print("\n--- Batch Features & Tensors ---")
for component, batch_dict in batch.items():
    print(f"\nComponent: [{component}]")
    for key, tensor in batch_dict.items():
        if isinstance(tensor, torch.Tensor):
            print(f"  {key}: tensor shape {tensor.shape}, dtype {tensor.dtype}")
        else:
            print(f"  {key}: {type(tensor).__name__}")